# Steps

- clean up social media text. 
- Then extract topics based on keywords
- generate sentiment score.
- transform social data into daily grain with daily mean sentiment score
- daily expected returns and risk measures (VaR, etc.) - via excel for efficiency
- combine new daily grain with daily mean sentiment score with daily stock data. Note: use pandas merge() or join().
- correlation coefficient between daily returns and sentiment score
- scatterplot between daily stock and daily mean sentiment score
- calculate various risk metrics 

# Setup

In [233]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from typing import Tuple, Dict, List
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, mean_absolute_percentage_error
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.linear_model import LinearRegression

In [234]:
# Custom
from utils.base import load_dataset
from utils.eda import check_dataset_size_and_shape, check_columns_for_nulls
from utils.tests import test_train_test_split_ratio

## Load Dataset

In [235]:
DATASET_FILENAME_1 = "wfc_socialmedia_data.csv"

In [236]:

df = load_dataset(DATASET_FILENAME_1)

## Shuffle Reproducabity

Without random_state, you'd get a different split each run, producing different train/test sets and therefore different model metrics each time. Passing RANDOM_STATE makes it reproducible. This applies to sklearn `train_test_split`.

`np.random.seed(RANDOM_STATE)` seeds NumPy's global RNG. It matters when using `np.random.*` functions directly (e.g., `np.random.shuffle`, `np.random.choice`).

In [237]:
RANDOM_STATE: int = 42  # global seed for full reproducibility
np.random.seed(RANDOM_STATE)

# Exploratory Data Analysis (EDA)

In [238]:
check_dataset_size_and_shape(df)

Number of rows: 384
Dataset shape: (384, 7)
Number of rows (alternative): 384


In [239]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 384 entries, 0 to 383
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   message_id       384 non-null    str    
 1   date             384 non-null    str    
 2   messageText      384 non-null    str    
 3   plainText        0 non-null      float64
 4   data_class       384 non-null    str    
 5   media_class      384 non-null    str    
 6   sentiment_score  0 non-null      float64
dtypes: float64(2), str(5)
memory usage: 21.1 KB


In [240]:
df.describe()

,plainText,sentiment_score
count,0.0,0.0
mean,NaN,NaN
std,NaN,NaN
min,NaN,NaN
25%,NaN,NaN
50%,NaN,NaN
75%,NaN,NaN
max,NaN,NaN


In [241]:
df.head()

,message_id,date,messageText,plainText,data_class,media_class,sentiment_score
0,7.69E+17,8/24/2016,"[<a class=""twitter-hashtag pretty-link js-nav""...",NaN,G,Twitter,NaN
1,7.69E+17,8/25/2016,"['Did ', <a class=""twitter-atreply pretty-link...",NaN,G,Twitter,NaN
2,7.69E+17,8/26/2016,"['Yall dont get it.Jenny from ', <strong>Wells...",NaN,G,Twitter,NaN
3,tAwIE9sJwPQ,8/26/2016,Most of the clowns I work with are using the L...,NaN,S,YouTube,NaN
4,tAwIE9sJwPQ,8/27/2016,<U+0001F44D><U+0001F3FC><U+0001F44D><U+0001F3F...,NaN,S,YouTube,NaN


## Missing Data

Expected it as placeholder columns to fill soon.


In [242]:
check_columns_for_nulls(df)

message_id         False
date               False
messageText        False
plainText           True
data_class         False
media_class        False
sentiment_score     True
dtype: bool


# Data Cleanup (preprocessing)

## Renaming

In [243]:
rename_map = {"messageText": "message_text", "plainText": "clean_text"}

df.rename(columns=rename_map, inplace=True)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 384 entries, 0 to 383
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   message_id       384 non-null    str    
 1   date             384 non-null    str    
 2   message_text     384 non-null    str    
 3   clean_text       0 non-null      float64
 4   data_class       384 non-null    str    
 5   media_class      384 non-null    str    
 6   sentiment_score  0 non-null      float64
dtypes: float64(2), str(5)
memory usage: 21.1 KB


## Removals (column drop)

In [244]:
# Remove columns using a list (safe: ignores names not present in df)
columns_to_remove = ["message_id", "data_class"]

# check columns in df before dropping inline
df.drop(columns=[col for col in columns_to_remove if col in df.columns], inplace=True)

df.head()

,date,message_text,clean_text,media_class,sentiment_score
0,8/24/2016,"[<a class=""twitter-hashtag pretty-link js-nav""...",NaN,Twitter,NaN
1,8/25/2016,"['Did ', <a class=""twitter-atreply pretty-link...",NaN,Twitter,NaN
2,8/26/2016,"['Yall dont get it.Jenny from ', <strong>Wells...",NaN,Twitter,NaN
3,8/26/2016,Most of the clowns I work with are using the L...,NaN,YouTube,NaN
4,8/27/2016,<U+0001F44D><U+0001F3FC><U+0001F44D><U+0001F3F...,NaN,YouTube,NaN


## clean_text

In [245]:
# For Ref but its replaced in the next cell since this was for text already in plain text.
# This was an initial attempt and end up working better for sentiment analysis since the text was already in plain text and maybe since it included spaces but not clear.

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)  # remove URLs
    text = re.sub(r'@\w+', '', text)             # remove @mentions
    text = re.sub(r'[^a-z\s]', '', text)         # letters only
    return text.strip()

df['clean_text_alt'] = df['message_text'].apply(clean_text)

df[['message_text', 'clean_text_alt']].head(10)

,message_text,clean_text_alt
0,"[<a class=""twitter-hashtag pretty-link js-nav""...",a classtwitterhashtag prettylink jsnav dataque...
1,"['Did ', <a class=""twitter-atreply pretty-link...",did a classtwitteratreply prettylink jsnav da...
2,"['Yall dont get it.Jenny from ', <strong>Wells...",yall dont get itjenny from strongwells fargos...
3,Most of the clowns I work with are using the L...,most of the clowns i work with are using the l...
4,<U+0001F44D><U+0001F3FC><U+0001F44D><U+0001F3F...,ufduffcufduffcufduffc
5,you and I are on the same page here. great video,you and i are on the same page here great video
6,['Student Ticketing opens at 8AM Saturday at '...,student ticketing opens at am saturday at str...
7,Great video ! Cheers from Belgium ! You rock m...,great video cheers from belgium you rock man...
8,"['San Mateo and Aragon pulling away in the ',...",san mateo and aragon pulling away in the a c...
9,"['Thank you to the ', <a class=""twitter-atrepl...",thank you to the a classtwitteratreply pretty...


In [246]:
# Remove spaces, whitespaces, hyperlinks, HTML tags and foreign language characters.

# Note: There may be a better way to do this that may support social media parsing better but this should do for sentiment analysis and other NLP tasks in this project.


def clean_message_text(text):
  text = str(text)
  text = re.sub(r'https?://\S+|www\.\S+', '', text)   # remove hyperlinks
  text = re.sub(r'<[^>]+>', '', text)                 # remove HTML tags
  text = text.encode('ascii', 'ignore').decode('ascii')  # remove foreign/non-ASCII chars
  text = re.sub(r'\s+', '', text)                     # remove spaces/whitespaces
  # return text
  return text.strip()

df["clean_text"] = df["message_text"].apply(clean_message_text)

df[["message_text", "clean_text"]].head(10)

,message_text,clean_text
0,"[<a class=""twitter-hashtag pretty-link js-nav""...","[#Irvine,'',#Jobs,'OperationsProcessor2:CA-Irv..."
1,"['Did ', <a class=""twitter-atreply pretty-link...","['Did',@wellsfargo,',',@fifththird,',',@AMD,'m..."
2,"['Yall dont get it.Jenny from ', <strong>Wells...","['Yalldontgetit.Jennyfrom',Wellsfargo,'isamort..."
3,Most of the clowns I work with are using the L...,MostoftheclownsIworkwithareusingtheLotteryasth...
4,<U+0001F44D><U+0001F3FC><U+0001F44D><U+0001F3F...,
5,you and I are on the same page here. great video,youandIareonthesamepagehere.greatvideo
6,['Student Ticketing opens at 8AM Saturday at '...,"['StudentTicketingopensat8AMSaturdayat',WellsF..."
7,Great video ! Cheers from Belgium ! You rock m...,Greatvideo!CheersfromBelgium!Yourockmanreallym...
8,"['San Mateo and Aragon pulling away in the ',...","['SanMateoandAragonpullingawayinthe',@WellsFar..."
9,"['Thank you to the ', <a class=""twitter-atrepl...","['Thankyoutothe',@WellsFargo,'teamthatisdedica..."


# Data Mining & Feature Engineering

Text mining via topic generation and feature engineering to create new columns before training (Minewiskan, n.d.). 

From AI:

"While data mining focuses on discovering hidden patterns in existing data, feature engineering focuses on preparing and modifying that data so machine learning models can understand it better. Data mining finds the insights; feature engineering builds the inputs."

## Topic Extraction

### Via Regex

Via regex and predefined keywords and sub keywords to look for.

In [247]:
# Then extract topics based on keywords

# Note: extract_topics works but its incompatible with the new clean_message_text function since it removes spaces and whitespaces. The normalize_for_match below is a better approach to extract topics based on keywords and is compatible with the new clean_message_text function. Kept here for reference but topics column gets overwritten by the new approach below.

# import re

topic_keywords = ["Wells Fargo", "opening fake accounts", "unauthorized accounts", "Scandal"]

def extract_topics(text: str):
  text = str(text).lower()
  matched = []
  for keyword in topic_keywords:
    if re.search(rf"\b{re.escape(keyword.lower())}\b", text):
      matched.append(keyword)
  return matched if matched else ["Other"]

df["topics"] = df["clean_text"].apply(extract_topics)

# Preview
df[["clean_text", "topics"]].head(10)

,clean_text,topics
0,"[#Irvine,'',#Jobs,'OperationsProcessor2:CA-Irv...",[Other]
1,"['Did',@wellsfargo,',',@fifththird,',',@AMD,'m...",[Other]
2,"['Yalldontgetit.Jennyfrom',Wellsfargo,'isamort...",[Other]
3,MostoftheclownsIworkwithareusingtheLotteryasth...,[Other]
4,,[Other]
5,youandIareonthesamepagehere.greatvideo,[Other]
6,"['StudentTicketingopensat8AMSaturdayat',WellsF...",[Other]
7,Greatvideo!CheersfromBelgium!Yourockmanreallym...,[Other]
8,"['SanMateoandAragonpullingawayinthe',@WellsFar...",[Other]
9,"['Thankyoutothe',@WellsFargo,'teamthatisdedica...",[Other]


In [248]:
# This method is more compatible with the new clean_message_text function since it normalizes the text and keywords for matching. It also allows for more flexible matching of keywords in the text. 
# Note: it could be refactored to be cleaner and maybe reusable on other cases.

topic_keywords = ["Wells Fargo", "opening fake accounts", "unauthorized accounts", "Scandal"]

def normalize_for_match(text):
  return re.sub(r"[^a-z0-9]+", "", str(text).lower())

normalized_keywords = {kw: normalize_for_match(kw) for kw in topic_keywords}

df["topics"] = df["clean_text"].apply(
  lambda text: [kw for kw, norm_kw in normalized_keywords.items() if norm_kw in normalize_for_match(text)] or ["Other"]
)

df[["clean_text", "topics"]].head(10)

,clean_text,topics
0,"[#Irvine,'',#Jobs,'OperationsProcessor2:CA-Irv...",[Wells Fargo]
1,"['Did',@wellsfargo,',',@fifththird,',',@AMD,'m...",[Wells Fargo]
2,"['Yalldontgetit.Jennyfrom',Wellsfargo,'isamort...",[Wells Fargo]
3,MostoftheclownsIworkwithareusingtheLotteryasth...,[Other]
4,,[Other]
5,youandIareonthesamepagehere.greatvideo,[Other]
6,"['StudentTicketingopensat8AMSaturdayat',WellsF...",[Wells Fargo]
7,Greatvideo!CheersfromBelgium!Yourockmanreallym...,[Other]
8,"['SanMateoandAragonpullingawayinthe',@WellsFar...",[Wells Fargo]
9,"['Thankyoutothe',@WellsFargo,'teamthatisdedica...",[Wells Fargo]


In [249]:
df["topics"].value_counts()

topics
[Other]                                 193
[Wells Fargo]                           172
[Wells Fargo, Scandal]                   15
[Wells Fargo, unauthorized accounts]      3
[Scandal]                                 1
Name: count, dtype: int64


### Via LDA (exploratory only)

Uses LDA (Latent Dirichlet Allocation), an unsupervised learning algorithm for hidden textual topic.

This is used to discover and validate the keywords used by the manual regex topic extraction above. Its not meant to be used to replace the manual extraction but in case there is a better keyword this is a way to find it.

In [250]:
import re
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

# def clean_text(text):
#     text = str(text).lower()
#     text = re.sub(r'http\S+|www\S+', '', text)  # remove URLs
#     text = re.sub(r'@\w+', '', text)             # remove @mentions
#     text = re.sub(r'[^a-z\s]', '', text)         # letters only
#     return text.strip()

def extract_topics_lda(df, text_column='clean_text', n_topics=5):
  # df['clean_text_lda'] = df[text_column].apply(clean_text)

  count_vec = CountVectorizer(stop_words='english', max_features=500, min_df=2)
  dtm = count_vec.fit_transform(df[text_column])

  lda = LatentDirichletAllocation(n_components=n_topics, random_state=42)
  lda.fit(dtm)

  feature_names = count_vec.get_feature_names_out()
  for idx, topic in enumerate(lda.components_):
      top_words = [feature_names[i] for i in topic.argsort()[:-9:-1]]
      print(f"Topic {idx+1}: {', '.join(top_words)}")

  df['topics_lda'] = lda.transform(dtm).argmax(axis=1)
  # return df[[text_column, 'topic_lda']].head(10)
  return df

df_lda = extract_topics_lda(df, n_topics=5)
df_lda[["clean_text", "topics_lda"]].head(10)

Topic 1: breaking, warren, lol, senate, trump, 21, buff, amp
Topic 2: twitter, data, dir, pre, ltr, true, aclass, embedded
Topic 3: wellsfargo, class, url, display, js, com, wells, fargo
Topic 4: wellsfargo, class, url, display, js, st, senwarren, fined
Topic 5: axlrose, center, greatvideo, rockorbust, wellsfargocenter, axldc, acdc, acdcphilly


,clean_text,topics_lda
0,"[#Irvine,'',#Jobs,'OperationsProcessor2:CA-Irv...",2
1,"['Did',@wellsfargo,',',@fifththird,',',@AMD,'m...",2
2,"['Yalldontgetit.Jennyfrom',Wellsfargo,'isamort...",2
3,MostoftheclownsIworkwithareusingtheLotteryasth...,0
4,,0
5,youandIareonthesamepagehere.greatvideo,4
6,"['StudentTicketingopensat8AMSaturdayat',WellsF...",2
7,Greatvideo!CheersfromBelgium!Yourockmanreallym...,4
8,"['SanMateoandAragonpullingawayinthe',@WellsFar...",2
9,"['Thankyoutothe',@WellsFargo,'teamthatisdedica...",1


## Removals (preprocessing - row drop)

In [251]:
# Remove rows where "topics" contains "Other"
df = df[~df["topics"].apply(lambda x: isinstance(x, list) and "Other" in x)].copy()

# Quick check
df["topics"].value_counts()

topics
[Wells Fargo]                           172
[Wells Fargo, Scandal]                   15
[Wells Fargo, unauthorized accounts]      3
[Scandal]                                 1
Name: count, dtype: int64

## Risk Measures (calculations)

### Set Impact Scores

In [252]:
# These impact scores were set arbitrarily for demonstration purposes. In a real-world scenario, they should be determined based on domain knowledge and risk assessment.

# Define impact scores for each topic (customize based on your risk assessment)
# Impact scores aligned to existing topic_keywords
impact_scores = {
  "Wells Fargo": 2,
  "opening fake accounts": 10,
  "unauthorized accounts": 10,
  "Scandal": 8,
  "Other": 1
}

# Optional guard: ensure every keyword has a score
missing_scores = [t for t in topic_keywords if t not in impact_scores]
if missing_scores:
  raise ValueError(f"Missing impact scores for: {missing_scores}")


# # Apply impact score per row (max impact when multiple topics exist)
# df["impact"] = df["topics"].apply(
#   lambda topics: max(impact_scores.get(t, 0) for t in topics) if isinstance(topics, list) and topics else 0
# )

# df[["topics", "impact"]].head(10)

### Impact, Likelyhood & Risk Factor

In [253]:
# Calculate likelihood (frequency of each topic)
topic_likelihood = df["topics"].explode().value_counts(normalize=True).to_dict()

# Calculate impact for each row
df["impact"] = df["topics"].apply(lambda topics: max([impact_scores.get(t, 0) for t in topics]))

# Calculate likelihood for each row (as a percentage)
df["likelihood"] = df["topics"].apply(lambda topics: max([topic_likelihood.get(t, 0) for t in topics]))

# Calculate risk factor (Likelihood × Impact)
df["risk_factor"] = df["likelihood"] * df["impact"]

print(df[["topics", "likelihood", "impact", "risk_factor"]].head(10))

           topics  likelihood  impact  risk_factor
0   [Wells Fargo]    0.909091       2     1.818182
1   [Wells Fargo]    0.909091       2     1.818182
2   [Wells Fargo]    0.909091       2     1.818182
6   [Wells Fargo]    0.909091       2     1.818182
8   [Wells Fargo]    0.909091       2     1.818182
9   [Wells Fargo]    0.909091       2     1.818182
10  [Wells Fargo]    0.909091       2     1.818182
12  [Wells Fargo]    0.909091       2     1.818182
13  [Wells Fargo]    0.909091       2     1.818182
15  [Wells Fargo]    0.909091       2     1.818182


In [254]:
df["impact"].value_counts()

impact
2     172
8      16
10      3
Name: count, dtype: int64

In [255]:
df["likelihood"].value_counts()

likelihood
0.909091    190
0.076555      1
Name: count, dtype: int64

In [256]:
df["risk_factor"].value_counts()

risk_factor
1.818182    172
7.272727     15
9.090909      3
0.612440      1
Name: count, dtype: int64

### Probability, Probability_Label & Risk Control Present

The Probability score indicates the estimated chance that a reputational risk event will occur for that row, on a scale from 0 to 1. So higher values mean more likely risk occurrence, and lower values mean less likely.

Also, probability_label indicates how likely the event is, while risk_control_present indicates whether mitigation controls are mentioned in the text.


In [257]:
# Probability: extract from clean text if present, otherwise infer from events, otherwise default to Medium
# This was set arbitrarily for demonstration purposes. In a real-world scenario, it should be determined based on domain knowledge and risk assessment.
probability_word_map = {
  "low": 0.30,
  "medium": 0.50,
  "high": 0.70,
}

def get_probability(row, text_column="clean_text"):
  text = str(row[text_column]).lower()

  # 1) Extract explicit percentage (e.g., "70%")
  pct_match = re.search(r'(\d+(?:\.\d+)?)\s*%', text)
  if pct_match:
    return max(0.0, min(1.0, float(pct_match.group(1)) / 100.0))

  # 2) Extract qualitative probability words
  for word, val in probability_word_map.items():
    if re.search(rf"\b{word}\b", text):
      return val

  # 3) Infer from mapped topic likelihood
  topics = row.get("topics", [])
  if isinstance(topics, list) and topics:
    inferred = [topic_likelihood.get(t) for t in topics if t in topic_likelihood]
    if inferred:
      return max(inferred)

  # 4) Default to Medium
  return 0.50

def probability_label(p):
  if p >= 0.67:
    return "High"
  if p >= 0.34:
    return "Medium"
  return "Low"

# Risk control presence: simple keyword-based detection
control_keywords = [
  "control", "mitigation", "backup", "redundancy", "monitoring", "alert",
  "patch", "firewall", "mfa", "encryption", "policy", "procedure",
  "training", "audit", "incident response", "access control"
]
# e.g. '\\b(control|mitigation|backup|r...'
control_pattern = r"\b(" + "|".join(re.escape(k) for k in control_keywords) + r")\b"

df["probability"] = df.apply(get_probability, axis=1)
df["probability_label"] = df["probability"].apply(probability_label)
df["risk_control_present"] = df["clean_text"].astype(str).str.lower().str.contains(control_pattern, regex=True)

df[["clean_text", "topics", "probability", "probability_label", "risk_control_present"]].head(10)

/var/folders/yh/2c6g86wn0_1_v36xltfm8dgh0000gn/T/ipykernel_28038/286272993.py:50: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df["risk_control_present"] = df["clean_text"].astype(str).str.lower().str.contains(control_pattern, regex=True)


,clean_text,topics,probability,probability_label,risk_control_present
0,"[#Irvine,'',#Jobs,'OperationsProcessor2:CA-Irv...",[Wells Fargo],0.909091,High,False
1,"['Did',@wellsfargo,',',@fifththird,',',@AMD,'m...",[Wells Fargo],0.909091,High,False
2,"['Yalldontgetit.Jennyfrom',Wellsfargo,'isamort...",[Wells Fargo],0.909091,High,False
6,"['StudentTicketingopensat8AMSaturdayat',WellsF...",[Wells Fargo],0.909091,High,False
8,"['SanMateoandAragonpullingawayinthe',@WellsFar...",[Wells Fargo],0.909091,High,False
9,"['Thankyoutothe',@WellsFargo,'teamthatisdedica...",[Wells Fargo],0.909091,High,False
10,"[WellsFargo,',Amazonenddealtoofferdiscountedst...",[Wells Fargo],0.909091,High,False
12,"['Thanks',@WellsFargo,'forworkwith',@OpGratitu...",[Wells Fargo],0.909091,High,False
13,"['.',@WellsFargo,'teamsupwith',@RamsNFL,'honor...",[Wells Fargo],0.909091,High,False
15,"['Nexttimeudiscourageachildfromlifeinthearts,I...",[Wells Fargo],0.909091,High,False


# Sentiment Analysis

In [258]:
# Note: clean_text did not perform well here so next cell overrides it with clean_text_alt which is a better approach for sentiment analysis since the text was already in plain text and maybe since it included spaces but not clear.

from nltk.sentiment.vader import SentimentIntensityAnalyzer
import nltk

nltk.download('vader_lexicon', quiet=True)

sia = SentimentIntensityAnalyzer()

def get_sentiment_scores(text: str) -> Dict[str, float]:
    scores = sia.polarity_scores(str(text))
    return scores['pos'], scores['neg'], scores['compound'], scores['neu']

df[['sa_positive', 'sa_negative', 'sa_sentiment_score', 'sa_neutral']] = df['clean_text'].apply(
    lambda x: pd.Series(get_sentiment_scores(x))
)

df[['clean_text', 'sa_positive', 'sa_negative', 'sa_sentiment_score', 'sa_neutral']].head(10)

,clean_text,sa_positive,sa_negative,sa_sentiment_score,sa_neutral
0,"[#Irvine,'',#Jobs,'OperationsProcessor2:CA-Irv...",0.0,0.0,0.0,1.0
1,"['Did',@wellsfargo,',',@fifththird,',',@AMD,'m...",0.0,0.0,0.0,1.0
2,"['Yalldontgetit.Jennyfrom',Wellsfargo,'isamort...",0.0,0.0,0.0,1.0
6,"['StudentTicketingopensat8AMSaturdayat',WellsF...",0.0,0.0,0.0,1.0
8,"['SanMateoandAragonpullingawayinthe',@WellsFar...",0.0,0.0,0.0,1.0
9,"['Thankyoutothe',@WellsFargo,'teamthatisdedica...",0.0,0.0,0.0,1.0
10,"[WellsFargo,',Amazonenddealtoofferdiscountedst...",0.0,0.0,0.0,1.0
12,"['Thanks',@WellsFargo,'forworkwith',@OpGratitu...",0.0,0.0,0.0,1.0
13,"['.',@WellsFargo,'teamsupwith',@RamsNFL,'honor...",0.0,0.0,0.0,1.0
15,"['Nexttimeudiscourageachildfromlifeinthearts,I...",0.0,0.0,0.0,1.0


In [259]:
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import nltk

nltk.download('vader_lexicon', quiet=True)

sia = SentimentIntensityAnalyzer()

def get_sentiment_scores(text: str) -> Dict[str, float]:
    scores = sia.polarity_scores(str(text))
    return scores['pos'], scores['neg'], scores['compound'], scores['neu']

df[['sa_positive', 'sa_negative', 'sa_sentiment_score', 'sa_neutral']] = df['clean_text_alt'].apply(
    lambda x: pd.Series(get_sentiment_scores(x))
)

df[['clean_text_alt', 'sa_positive', 'sa_negative', 'sa_sentiment_score', 'sa_neutral']].head(10)

,clean_text_alt,sa_positive,sa_negative,sa_sentiment_score,sa_neutral
0,a classtwitterhashtag prettylink jsnav dataque...,0.000,0.000,0.0000,1.000
1,did a classtwitteratreply prettylink jsnav da...,0.065,0.000,0.5719,0.935
2,yall dont get itjenny from strongwells fargos...,0.000,0.278,-0.8271,0.722
6,student ticketing opens at am saturday at str...,0.000,0.000,0.0000,1.000
8,san mateo and aragon pulling away in the a c...,0.000,0.000,0.0000,1.000
9,thank you to the a classtwitteratreply pretty...,0.175,0.000,0.6705,0.825
10,strongwells fargostrong amazon end deal to of...,0.094,0.000,0.2263,0.906
12,thanks a classtwitteratreply prettylink jsnav...,0.086,0.000,0.4404,0.914
13,a classtwitteratreply prettylink jsnav datamen...,0.068,0.000,0.5106,0.932
15,next time u discourage a child from life in th...,0.306,0.075,0.8225,0.618


## Check Counts (distribution)

TODO: graph it via histogram and write a test to help catch if the clean up step needs to be improved. Or consider other more complex alternatives.

In [260]:
df['sa_sentiment_score'].value_counts()

sa_sentiment_score
 0.0000    50
-0.1027     5
-0.5859     4
-0.6486     4
-0.3400     4
           ..
-0.7783     1
 0.4939     1
-0.6249     1
-0.5267     1
-0.4874     1
Name: count, Length: 90, dtype: int64

# Preprocessing

In [261]:
df.info()

<class 'pandas.DataFrame'>
Index: 191 entries, 0 to 378
Data columns (total 18 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   date                  191 non-null    str    
 1   message_text          191 non-null    str    
 2   clean_text            191 non-null    str    
 3   media_class           191 non-null    str    
 4   sentiment_score       0 non-null      float64
 5   clean_text_alt        191 non-null    str    
 6   topics                191 non-null    object 
 7   topics_lda            191 non-null    int64  
 8   impact                191 non-null    int64  
 9   likelihood            191 non-null    float64
 10  risk_factor           191 non-null    float64
 11  probability           191 non-null    float64
 12  probability_label     191 non-null    str    
 13  risk_control_present  191 non-null    bool   
 14  sa_positive           191 non-null    float64
 15  sa_negative           191 non-null    f

In [262]:
df.head()

,date,message_text,clean_text,media_class,sentiment_score,clean_text_alt,topics,topics_lda,impact,likelihood,risk_factor,probability,probability_label,risk_control_present,sa_positive,sa_negative,sa_sentiment_score,sa_neutral
0,8/24/2016,"[<a class=""twitter-hashtag pretty-link js-nav""...","[#Irvine,'',#Jobs,'OperationsProcessor2:CA-Irv...",Twitter,NaN,a classtwitterhashtag prettylink jsnav dataque...,[Wells Fargo],2,2,0.909091,1.818182,0.909091,High,False,0.000,0.000,0.0000,1.000
1,8/25/2016,"['Did ', <a class=""twitter-atreply pretty-link...","['Did',@wellsfargo,',',@fifththird,',',@AMD,'m...",Twitter,NaN,did a classtwitteratreply prettylink jsnav da...,[Wells Fargo],2,2,0.909091,1.818182,0.909091,High,False,0.065,0.000,0.5719,0.935
2,8/26/2016,"['Yall dont get it.Jenny from ', <strong>Wells...","['Yalldontgetit.Jennyfrom',Wellsfargo,'isamort...",Twitter,NaN,yall dont get itjenny from strongwells fargos...,[Wells Fargo],2,2,0.909091,1.818182,0.909091,High,False,0.000,0.278,-0.8271,0.722
6,8/29/2016,['Student Ticketing opens at 8AM Saturday at '...,"['StudentTicketingopensat8AMSaturdayat',WellsF...",Twitter,NaN,student ticketing opens at am saturday at str...,[Wells Fargo],2,2,0.909091,1.818182,0.909091,High,False,0.000,0.000,0.0000,1.000
8,8/30/2016,"['San Mateo and Aragon pulling away in the ',...","['SanMateoandAragonpullingawayinthe',@WellsFar...",Twitter,NaN,san mateo and aragon pulling away in the a c...,[Wells Fargo],2,2,0.909091,1.818182,0.909091,High,False,0.000,0.000,0.0000,1.000


## Renaming (optional)

In [263]:
# rename_map = {"Topics": "events", "Subtopics": "subevents"}

# df.rename(columns=rename_map, inplace=True)
# df.info()

# unless there is a need
pass

In [264]:
# df[["events", "subevents"]].head()

## Encoding Categaricals

In [265]:
# get all string or object columns in the dataframe
string_cols = df.select_dtypes(include=["object", "string"]).columns.tolist()
print("String/Object columns in the dataframe:", string_cols)

# set columns to covert/encode manually
manual_encode_cols = ["topics", "probability_label","media_class"]
print("Columns to encode manually:", manual_encode_cols)

String/Object columns in the dataframe: ['date', 'message_text', 'clean_text', 'media_class', 'clean_text_alt', 'topics', 'probability_label']
Columns to encode manually: ['topics', 'probability_label', 'media_class']


In [266]:
from sklearn.preprocessing import OrdinalEncoder

# this catches other types of categorical data (lists, dicts) and normalizes them to strings before encoding
# Note: there might be a better ways to handle this.
def normalize_categorical_value(v):
    if isinstance(v, list):
        return " | ".join(map(str, v))
    if isinstance(v, dict):
        return " | ".join(f"{k}:{v[k]}" for k in sorted(v))
    return "" if v is None else str(v)

def encode_categorical_columns(df, columns):
  for col in columns:
      df[col] = df[col].apply(normalize_categorical_value)

  if columns:
      enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
      df[columns] = enc.fit_transform(df[columns])
  return df

df = encode_categorical_columns(df, manual_encode_cols)

In [267]:
df.info()

<class 'pandas.DataFrame'>
Index: 191 entries, 0 to 378
Data columns (total 18 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   date                  191 non-null    str    
 1   message_text          191 non-null    str    
 2   clean_text            191 non-null    str    
 3   media_class           191 non-null    float64
 4   sentiment_score       0 non-null      float64
 5   clean_text_alt        191 non-null    str    
 6   topics                191 non-null    float64
 7   topics_lda            191 non-null    int64  
 8   impact                191 non-null    int64  
 9   likelihood            191 non-null    float64
 10  risk_factor           191 non-null    float64
 11  probability           191 non-null    float64
 12  probability_label     191 non-null    float64
 13  risk_control_present  191 non-null    bool   
 14  sa_positive           191 non-null    float64
 15  sa_negative           191 non-null    f

In [268]:
df.head()

,date,message_text,clean_text,media_class,sentiment_score,clean_text_alt,topics,topics_lda,impact,likelihood,risk_factor,probability,probability_label,risk_control_present,sa_positive,sa_negative,sa_sentiment_score,sa_neutral
0,8/24/2016,"[<a class=""twitter-hashtag pretty-link js-nav""...","[#Irvine,'',#Jobs,'OperationsProcessor2:CA-Irv...",0.0,NaN,a classtwitterhashtag prettylink jsnav dataque...,1.0,2,2,0.909091,1.818182,0.909091,0.0,False,0.000,0.000,0.0000,1.000
1,8/25/2016,"['Did ', <a class=""twitter-atreply pretty-link...","['Did',@wellsfargo,',',@fifththird,',',@AMD,'m...",0.0,NaN,did a classtwitteratreply prettylink jsnav da...,1.0,2,2,0.909091,1.818182,0.909091,0.0,False,0.065,0.000,0.5719,0.935
2,8/26/2016,"['Yall dont get it.Jenny from ', <strong>Wells...","['Yalldontgetit.Jennyfrom',Wellsfargo,'isamort...",0.0,NaN,yall dont get itjenny from strongwells fargos...,1.0,2,2,0.909091,1.818182,0.909091,0.0,False,0.000,0.278,-0.8271,0.722
6,8/29/2016,['Student Ticketing opens at 8AM Saturday at '...,"['StudentTicketingopensat8AMSaturdayat',WellsF...",0.0,NaN,student ticketing opens at am saturday at str...,1.0,2,2,0.909091,1.818182,0.909091,0.0,False,0.000,0.000,0.0000,1.000
8,8/30/2016,"['San Mateo and Aragon pulling away in the ',...","['SanMateoandAragonpullingawayinthe',@WellsFar...",0.0,NaN,san mateo and aragon pulling away in the a c...,1.0,2,2,0.909091,1.818182,0.909091,0.0,False,0.000,0.000,0.0000,1.000


### Categorical Check (test)

In [269]:
# Gen via GPT-5.3-Codex.
# check for any remaining list/dict/tuple/set values in object/string columns that would break sklearn encoders
# Use this validation cell to confirm there are no list/dict values left in categorical columns before encoding

import pandas as pd

obj_cols = df.select_dtypes(include=["object", "string"]).columns.tolist()

def is_container_like(x):
    return isinstance(x, (list, dict, tuple, set))

issues = []
for c in obj_cols:
    mask = df[c].apply(is_container_like)
    n = int(mask.sum())
    if n > 0:
        sample = df.loc[mask, c].head(3).tolist()
        issues.append({"column": c, "rows_with_container_values": n, "sample_values": sample})

if issues:
    print("Columns that will break sklearn encoders:")
    print(pd.DataFrame(issues))
else:
    print("OK: No list/dict/tuple/set values found in object/string columns.")

OK: No list/dict/tuple/set values found in object/string columns.


## Date Conversion

From AI:

Three strategies.

1. Extract Cyclical and Categorical Features. This is the best method for forecasting because it extracts specific time components that models can learn from.

2. Convert to a Unix Timestamp. If you want the model to see time as a continuous, linear timeline, convert the dates into seconds or nanoseconds elapsed since January 1, 1970.

3. Set the Date as the Index. If you are building an actual time-series model (like ARIMA, Prophet, or certain LSTM setups), remove the date from the training columns entirely and place it into the index.

Knowing the algorithm can help determine which of these feature extraction steps will work best.

In [270]:

# 1. Extract Cyclical and Categorical Features.

# Convert to datetime first
# Best for merging
df['date_ob'] = pd.to_datetime(df['date'])

# Extract numerical components
df['year'] = df['date_ob'].dt.year
df['month'] = df['date_ob'].dt.month
df['day'] = df['date_ob'].dt.day
df['day_of_week'] = df['date_ob'].dt.dayofweek  # 0 is Monday, 6 is Sunday
df['is_weekend'] = df['date_ob'].dt.dayofweek.isin([5, 6]).astype(int)

# 2. Convert to a Unix Timestamp.
# Convert datetime to a raw numerical integer
df['timestamp_numeric'] = df['date_ob'].astype('int64')

# 3. Set the Date as the Index.
# Move the datetime out of the features and into the index
# df.set_index('date_ob', inplace=True)

In [271]:
df.head()

,date,message_text,clean_text,media_class,sentiment_score,clean_text_alt,topics,topics_lda,impact,likelihood,...,sa_negative,sa_sentiment_score,sa_neutral,date_ob,year,month,day,day_of_week,is_weekend,timestamp_numeric
0,8/24/2016,"[<a class=""twitter-hashtag pretty-link js-nav""...","[#Irvine,'',#Jobs,'OperationsProcessor2:CA-Irv...",0.0,NaN,a classtwitterhashtag prettylink jsnav dataque...,1.0,2,2,0.909091,...,0.000,0.0000,1.000,2016-08-24,2016,8,24,2,0,1471996800000000
1,8/25/2016,"['Did ', <a class=""twitter-atreply pretty-link...","['Did',@wellsfargo,',',@fifththird,',',@AMD,'m...",0.0,NaN,did a classtwitteratreply prettylink jsnav da...,1.0,2,2,0.909091,...,0.000,0.5719,0.935,2016-08-25,2016,8,25,3,0,1472083200000000
2,8/26/2016,"['Yall dont get it.Jenny from ', <strong>Wells...","['Yalldontgetit.Jennyfrom',Wellsfargo,'isamort...",0.0,NaN,yall dont get itjenny from strongwells fargos...,1.0,2,2,0.909091,...,0.278,-0.8271,0.722,2016-08-26,2016,8,26,4,0,1472169600000000
6,8/29/2016,['Student Ticketing opens at 8AM Saturday at '...,"['StudentTicketingopensat8AMSaturdayat',WellsF...",0.0,NaN,student ticketing opens at am saturday at str...,1.0,2,2,0.909091,...,0.000,0.0000,1.000,2016-08-29,2016,8,29,0,0,1472428800000000
8,8/30/2016,"['San Mateo and Aragon pulling away in the ',...","['SanMateoandAragonpullingawayinthe',@WellsFar...",0.0,NaN,san mateo and aragon pulling away in the a c...,1.0,2,2,0.909091,...,0.000,0.0000,1.000,2016-08-30,2016,8,30,1,0,1472515200000000


### Check Day Counts

Important as this dataset will be merged at a day grain with stock dataset. Later step will aggregate per mean sentiment score to transform grain. Best to check full date rather than just day.

In [272]:
df["date"].value_counts()

date
9/29/2016    17
9/8/2016     13
9/21/2016    13
9/9/2016     11
9/28/2016    11
9/30/2016     9
9/13/2016     8
9/27/2016     8
9/22/2016     7
9/14/2016     6
9/16/2016     6
9/20/2016     6
9/12/2016     5
9/23/2016     5
9/26/2016     5
10/3/2016     5
9/3/2016      4
9/15/2016     4
9/17/2016     3
9/19/2016     3
9/25/2016     3
10/1/2016     3
10/4/2016     3
10/5/2016     3
10/6/2016     3
8/31/2016     2
9/10/2016     2
9/24/2016     2
10/2/2016     2
10/7/2016     2
10/8/2016     2
10/9/2016     2
8/24/2016     1
8/25/2016     1
8/26/2016     1
8/29/2016     1
8/30/2016     1
9/1/2016      1
9/2/2016      1
9/4/2016      1
9/5/2016      1
9/6/2016      1
9/7/2016      1
9/11/2016     1
9/18/2016     1
Name: count, dtype: int64

In [275]:
date_value_counts = df["date_ob"].value_counts().sort_index()
print(date_value_counts)

date_ob
2016-08-24     1
2016-08-25     1
2016-08-26     1
2016-08-29     1
2016-08-30     1
2016-08-31     2
2016-09-01     1
2016-09-02     1
2016-09-03     4
2016-09-04     1
2016-09-05     1
2016-09-06     1
2016-09-07     1
2016-09-08    13
2016-09-09    11
2016-09-10     2
2016-09-11     1
2016-09-12     5
2016-09-13     8
2016-09-14     6
2016-09-15     4
2016-09-16     6
2016-09-17     3
2016-09-18     1
2016-09-19     3
2016-09-20     6
2016-09-21    13
2016-09-22     7
2016-09-23     5
2016-09-24     2
2016-09-25     3
2016-09-26     5
2016-09-27     8
2016-09-28    11
2016-09-29    17
2016-09-30     9
2016-10-01     3
2016-10-02     2
2016-10-03     5
2016-10-04     3
2016-10-05     3
2016-10-06     3
2016-10-07     2
2016-10-08     2
2016-10-09     2
Name: count, dtype: int64


# New Dataset - Semantic Analysis Daily Grain (transformation)

In [274]:
# Daily mean sentiment score without overwriting df
if "date_ob" in df.columns:
  date_series = pd.to_datetime(df["date_ob"])
elif "date" in df.columns:
  date_series = pd.to_datetime(df["date"])
elif "timestamp_numeric" in df.columns:
  date_series = pd.to_datetime(df["timestamp_numeric"])
else:
  date_series = pd.to_datetime(
    {"year": df["year"], "month": df["month"], "day": df["day"]}
  )

daily_sentiment = (
  df.assign(date_ob=date_series.dt.normalize())
    .groupby("date_ob", as_index=False)
    .agg(
      daily_mean_sentiment_score=("sa_sentiment_score", "mean"),
      daily_post_count=("sa_sentiment_score", "size")
    )
    .sort_values("date_ob")
)

daily_sentiment.head(10)

,date_ob,daily_mean_sentiment_score,daily_post_count
0,2016-08-24,0.000000,1
1,2016-08-25,0.571900,1
2,2016-08-26,-0.827100,1
3,2016-08-29,0.000000,1
4,2016-08-30,0.000000,1
5,2016-08-31,0.448400,2
6,2016-09-01,0.440400,1
7,2016-09-02,0.510600,1
8,2016-09-03,0.077975,4
9,2016-09-04,0.441700,1


## Merge Dataset (OPTIONAL - grain transformation)

May be needed in the future but daily_sentiment is enough to merge with stock daily dataset.

In [ ]:
# To add more columns, extend .agg(...) with how each column should collapse:
# daily_sentiment = (
#   df.assign(date_ob=date_series.dt.normalize())
#     .groupby("date_ob", as_index=False)
#     .agg(
#       daily_mean_sentiment_score=("sa_sentiment_score", "mean"),
#       daily_post_count=("sa_sentiment_score", "size"),
#       daily_mean_risk_factor=("risk_factor", "mean"),
#       daily_mean_probability=("probability", "mean"),
#       max_impact=("impact", "max"),
#       avg_likelihood=("likelihood", "mean"),
#       # categorical examples:
#       top_probability_label=("probability_label", lambda s: s.mode().iat[0] if not s.mode().empty else None),
#       any_risk_control_present=("risk_control_present", "max")
#     )
#     .sort_values("date_ob")
# )

# or consider merging back into df row grain
# # 1) Build a stable merge key in df (same grain/type as daily_sentiment.date_ob)
# df["date_merge"] = pd.to_datetime(df["date_ob"]).dt.normalize()

# # 2) Ensure daily_sentiment has the same key type
# daily_sentiment["date_ob"] = pd.to_datetime(daily_sentiment["date_ob"]).dt.normalize()

# # 3) Merge daily features back to row-level df
# df = df.merge(
#     daily_sentiment,
#     left_on="date_merge",
#     right_on="date_ob",
#     how="left",
#     validate="many_to_one"  # many df rows per day -> one daily row
# )

# # 4) Optional cleanup
# df.drop(columns=["date_merge", "date_ob_y"], errors="ignore", inplace=True)
# df.rename(columns={"date_ob_x": "date_ob"}, inplace=True)

# df.head()

## Removals (final)

In [ ]:
# Remove columns using a list (safe: ignores names not present in df)
columns_to_remove = ["message_text", "clean_text", "date", "date_ob"]

# check columns in df before dropping inline
df.drop(columns=[col for col in columns_to_remove if col in df.columns], inplace=True)

df.head()

,media_class,sentiment_score,clean_text_alt,topics,topics_lda,impact,likelihood,risk_factor,probability,probability_label,...,sa_positive,sa_negative,sa_sentiment_score,sa_neutral,year,month,day,day_of_week,is_weekend,timestamp_numeric
0,0.0,NaN,a classtwitterhashtag prettylink jsnav dataque...,1.0,2,2,0.909091,1.818182,0.909091,0.0,...,0.000,0.000,0.0000,1.000,2016,8,24,2,0,1471996800000000
1,0.0,NaN,did a classtwitteratreply prettylink jsnav da...,1.0,2,2,0.909091,1.818182,0.909091,0.0,...,0.065,0.000,0.5719,0.935,2016,8,25,3,0,1472083200000000
2,0.0,NaN,yall dont get itjenny from strongwells fargos...,1.0,2,2,0.909091,1.818182,0.909091,0.0,...,0.000,0.278,-0.8271,0.722,2016,8,26,4,0,1472169600000000
6,0.0,NaN,student ticketing opens at am saturday at str...,1.0,2,2,0.909091,1.818182,0.909091,0.0,...,0.000,0.000,0.0000,1.000,2016,8,29,0,0,1472428800000000
8,0.0,NaN,san mateo and aragon pulling away in the a c...,1.0,2,2,0.909091,1.818182,0.909091,0.0,...,0.000,0.000,0.0000,1.000,2016,8,30,1,0,1472515200000000


# Exploratory Data Analysis (EDA)

In [ ]:
df.info()

<class 'pandas.DataFrame'>
Index: 191 entries, 0 to 378
Data columns (total 21 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   media_class           191 non-null    float64
 1   sentiment_score       0 non-null      float64
 2   clean_text_alt        191 non-null    str    
 3   topics                191 non-null    float64
 4   topics_lda            191 non-null    int64  
 5   impact                191 non-null    int64  
 6   likelihood            191 non-null    float64
 7   risk_factor           191 non-null    float64
 8   probability           191 non-null    float64
 9   probability_label     191 non-null    float64
 10  risk_control_present  191 non-null    bool   
 11  sa_positive           191 non-null    float64
 12  sa_negative           191 non-null    float64
 13  sa_sentiment_score    191 non-null    float64
 14  sa_neutral            191 non-null    float64
 15  year                  191 non-null    i

In [ ]:
df.describe()

,media_class,sentiment_score,topics,topics_lda,impact,likelihood,risk_factor,probability,probability_label,sa_positive,sa_negative,sa_sentiment_score,sa_neutral,year,month,day,day_of_week,is_weekend,timestamp_numeric
count,191.000000,0.0,191.000000,191.000000,191.000000,191.000000,191.000000,191.000000,191.000000,191.000000,191.000000,191.000000,191.000000,191.0,191.000000,191.000000,191.000000,191.000000,1.910000e+02
mean,0.099476,NaN,1.104712,1.596859,2.628272,0.904732,2.354468,0.895694,0.015707,0.051084,0.076670,-0.136562,0.872246,2016.0,9.094241,17.350785,2.717277,0.136126,1.474342e+15
std,0.300087,NaN,0.369251,0.732545,1.909267,0.060240,1.703681,0.097170,0.124665,0.082086,0.087589,0.421096,0.122267,0.0,0.399366,9.264959,1.620158,0.343823,8.930949e+11
min,0.000000,NaN,0.000000,0.000000,2.000000,0.076555,0.612440,0.076555,0.000000,0.000000,0.000000,-0.965100,0.154000,2016.0,8.000000,1.000000,0.000000,0.000000,1.471997e+15
25%,0.000000,NaN,1.000000,1.000000,2.000000,0.909091,1.818182,0.909091,0.000000,0.000000,0.000000,-0.493900,0.803000,2016.0,9.000000,9.000000,2.000000,0.000000,1.473638e+15
50%,0.000000,NaN,1.000000,2.000000,2.000000,0.909091,1.818182,0.909091,0.000000,0.000000,0.061000,0.000000,0.890000,2016.0,9.000000,19.000000,3.000000,0.000000,1.474416e+15
75%,0.000000,NaN,1.000000,2.000000,2.000000,0.909091,1.818182,0.909091,0.000000,0.081000,0.134500,0.000000,0.987500,2016.0,9.000000,27.000000,4.000000,0.000000,1.475107e+15
max,1.000000,NaN,3.000000,4.000000,10.000000,0.909091,9.090909,0.909091,1.000000,0.451000,0.538000,0.883400,1.000000,2016.0,10.000000,31.000000,6.000000,1.000000,1.475971e+15


In [ ]:
df.head()

,media_class,sentiment_score,clean_text_alt,topics,topics_lda,impact,likelihood,risk_factor,probability,probability_label,...,sa_positive,sa_negative,sa_sentiment_score,sa_neutral,year,month,day,day_of_week,is_weekend,timestamp_numeric
0,0.0,NaN,a classtwitterhashtag prettylink jsnav dataque...,1.0,2,2,0.909091,1.818182,0.909091,0.0,...,0.000,0.000,0.0000,1.000,2016,8,24,2,0,1471996800000000
1,0.0,NaN,did a classtwitteratreply prettylink jsnav da...,1.0,2,2,0.909091,1.818182,0.909091,0.0,...,0.065,0.000,0.5719,0.935,2016,8,25,3,0,1472083200000000
2,0.0,NaN,yall dont get itjenny from strongwells fargos...,1.0,2,2,0.909091,1.818182,0.909091,0.0,...,0.000,0.278,-0.8271,0.722,2016,8,26,4,0,1472169600000000
6,0.0,NaN,student ticketing opens at am saturday at str...,1.0,2,2,0.909091,1.818182,0.909091,0.0,...,0.000,0.000,0.0000,1.000,2016,8,29,0,0,1472428800000000
8,0.0,NaN,san mateo and aragon pulling away in the a c...,1.0,2,2,0.909091,1.818182,0.909091,0.0,...,0.000,0.000,0.0000,1.000,2016,8,30,1,0,1472515200000000


# ERROR (temp stop for future model training)

In [ ]:
raise NotImplementedError("This section of the code is not yet implemented.")

NotImplementedError: This section of the code is not yet implemented.

# Normalization (optional)

# Train/Test Split

In [ ]:
# get pd.Series/Vector for predictors
# ref @ https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.drop.html
# Assumes numeric columns only after dropping non-numeric ones earlier in feature selection
X = df.drop(columns=['Risk_factor'])

# get pd.Series/Vector for target
y = df['Risk_factor']

# split 80/20
# per https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)


In [ ]:
test_train_test_split_ratio(df, X_train, X_test, expected_test_size=0.2)

Number of rows: 20
Dataset shape: (20, 11)
Number of rows (alternative): 20
Train size: 16, Test size: 4, Test ratio: 20.00%


In [ ]:
X_train.info()

<class 'pandas.DataFrame'>
Index: 16 entries, 8 to 6
Data columns (total 10 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Emails                16 non-null     float64
 1   events                16 non-null     float64
 2   subevents             16 non-null     float64
 3   clean_text_lda        16 non-null     float64
 4   topic_lda             16 non-null     int64  
 5   Impact                16 non-null     int64  
 6   Likelihood            16 non-null     float64
 7   Probability           16 non-null     float64
 8   Probability_Label     16 non-null     float64
 9   Risk Control Present  16 non-null     bool   
dtypes: bool(1), float64(7), int64(2)
memory usage: 1.3 KB


# Training

## Linear Regression

In [ ]:
# https://scikit-learn.org/stable/auto_examples/linear_model/plot_ols_ridge.html
regressor = LinearRegression().fit(X_train, y_train)
# y_pred = regressor.predict(X_test)

## Support Vector Machine

Via a regressor model (SVR) since target is continous and not categorical.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler

svr = make_pipeline(
    StandardScaler(),
    SVR(kernel="rbf", C=1.0, epsilon=0.1)
)

svr.fit(X_train, y_train)
# y_pred = svr.predict(X_test)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('standardscaler', ...), ('svr', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](10,)","['Emails','events','subevents',...,'Probability','Probability_Label', 'Risk Control Present']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,10
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
Name,Type,Value


# Evaluation

## For Linear Regression 

In [ ]:

def evaluate_model(model, X_test, y_test):
  y_pred = model.predict(X_test)
  
  mse = mean_squared_error(y_test, y_pred)
  rmse = np.sqrt(mse)
  mae = mean_absolute_error(y_test, y_pred)
  r2 = r2_score(y_test, y_pred)
  mape = mean_absolute_percentage_error(y_test, y_pred)

  test_mse = float(round(mse,3))
  test_rmse = float(round(rmse,3))
  test_mae = float(round(mae,3))
  test_r2 = float(round(r2,3))
  test_mape = float(round(mape * 100, 3))  # convert to percentage
  
  # return test_mse, test_mae, test_r2
  return {
    'mse': test_mse,
    'rmse': test_rmse,
    'mae': test_mae,
    'r2': test_r2,
    'mape': test_mape,
  }

metrics = evaluate_model(regressor, X_test, y_test)
print(f"Test MSE: {metrics['mse']}")
print(f"Test RMSE: {metrics['rmse']}")
print(f"Test MAE: {metrics['mae']}")
print(f"Test R²: {metrics['r2']}")
print(f"Test MAPE: {metrics['mape']}%")

Test MSE: 3.301
Test RMSE: 1.817
Test MAE: 0.908
Test R²: -40.668
Test MAPE: 67.29%


In [ ]:
metrics_svr = evaluate_model(svr, X_test, y_test)
print(f"Test MSE: {metrics_svr['mse']}")
print(f"Test RMSE: {metrics_svr['rmse']}")
print(f"Test MAE: {metrics_svr['mae']}")
print(f"Test R²: {metrics_svr['r2']}")
print(f"Test MAPE: {metrics_svr['mape']}%")

Test MSE: 0.008
Test RMSE: 0.088
Test MAE: 0.08
Test R²: 0.902
Test MAPE: 5.098%


# References

